# Brain Stroke CT Baseline

The notebook is self-contained: it extracts the embedded source bundle into
`/kaggle/working`, then runs training, evaluation, and external inference.

Progress appears live in the training cell and is persisted in `run.log` and
`progress.json`.

In [ ]:
from pathlib import Path
import base64
import io
import os
import shutil
import subprocess
import sys
import tarfile

repo = Path('/kaggle/working/brain-stroke-ct-segmentation')
if repo.exists():
    shutil.rmtree(repo)
repo.mkdir(parents=True)

payload = '...'
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as archive:
    archive.extractall(repo, filter='data')

candidates = [
    path for path in Path('/kaggle/input').rglob('Brain_Stroke_CT_Dataset')
    if all((path / name).is_dir() for name in ('Bleeding', 'Ischemia', 'Normal', 'External_Test'))
]
if not candidates:
    raise FileNotFoundError('Brain_Stroke_CT_Dataset was not mounted')
data_root = candidates[0]
os.environ['BSC_DATA_ROOT'] = str(data_root)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
print('repository:', repo)
print('dataset:', data_root)
print('python:', sys.version)

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/train.py', '--config', 'configs/baseline.yaml'],
    cwd=repo,
    check=True,
    env=os.environ.copy(),
)

In [ ]:
subprocess.run(
    [sys.executable, 'scripts/evaluate.py', '--config', 'configs/baseline.yaml'],
    cwd=repo,
    check=True,
    env=os.environ.copy(),
)
subprocess.run(
    [sys.executable, 'scripts/infer.py', '--config', 'configs/baseline.yaml'],
    cwd=repo,
    check=True,
    env=os.environ.copy(),
)

In [ ]:
results = repo / 'reports' / 'experiments' / 'baseline'
print((results / 'summary.md').read_text(encoding='utf-8'))
(Path('/kaggle/working') / 'BASELINE_RUN_COMPLETE').write_text('complete\n', encoding='utf-8')
print('result directory:', results)